# 01_2 Synthetic slides: from the tile embeddings of the real slides to saved synthetic vectors

**Overview.** This notebook turns the tile embeddings saved by `01_wsi_to_vectors.ipynb` into **synthetic slides** and saves them as feature files of the same format, together with a label file, so that the classifier notebook (`02_3_mod_vector_to_classifier_finetunning.ipynb`) only has to load them. The classes are Non-cancer (N, PB, UDH, FEA), non-invasive atypia/neoplasia (ADH and DCIS) and invasive carcinoma (IC).

A synthetic slide is a bag of 4096 or 8192 tile embeddings drawn at random, with replacement, from real slides of one class. It takes tiles from at least `K_SLIDES` different real slides, no real slide gives more than `TOP_PERCENTAGE` of its tiles, and the tiles are put in a random order. The official Training and Validation slides are pooled and divided into patient-grouped, class-stratified folds (`utils/mil_cv.py`). For every fold, `N_TOTAL` synthetic slides are made, balanced over the classes: `EARLY_STOP_FRACTION` of them from the real early-stopping slides of the fold, the others from its real fit slides. The held-out slides of the fold and the official Testing slides are never a source, so a model trained on the synthetic slides of a fold has seen no tile of the patients that are scored with it. Because of this the synthetic slides are saved per fold.

The vectors are written to `OUT_DIR/UNI2-h/fold_<k>/<slide>.pt` and the labels to `labels/BRACS_mod.csv` (CSV rather than xlsx: no writer for xlsx exists in the project and no package may be added).

```mermaid
flowchart TD
    subgraph IN["Inputs"]
        FEAT[("level_1/UNI2-h/<br/>one .pt file per real slide (from 01)")]
        LAB[("labels/BRACS.xlsx<br/>class, patient, official set")]
    end

    subgraph S3["3. Data"]
        D1["3.1 build_slide_table + load_bags"]
        D2["3.2 relabel_table<br/>Non-cancer / atypia-neoplasia / IC"]
        D3["3.3 make_cv_folds<br/>patient-grouped, stratified"]
    end

    subgraph S4["4. Synthetic slides"]
        R1["4.1 make_synthetic_folds<br/>N_TOTAL recipes per fold,<br/>fit and early-stopping slides only"]
        R2["4.2 plot_synthetic_summary"]
    end

    subgraph S5["5. Saving"]
        W1["save_synthetic_folds<br/>one .pt per synthetic slide"]
        W2["write_mod_labels"]
    end

    subgraph S6["6. Verification"]
        V1["reload + checks<br/>tiles, sources, folds"]
    end

    REC[("level_1_mod/recipes/<br/>synthetic_recipes.pt")]
    OUT[("level_1_mod/UNI2-h/fold_k/<br/>SYN_*.pt")]
    CSV[("labels/BRACS_mod.csv")]

    FEAT --> D1
    LAB --> D1
    D1 --> D2
    D2 --> D3
    D3 --> R1
    R1 --> REC
    R1 --> R2
    R1 --> W1
    R1 --> W2
    W1 --> OUT
    W2 --> CSV
    OUT --> V1
    CSV --> V1
```

## 1. Setup

The environment is recorded so the run can be reproduced.

In [ ]:
# STEP 1: Standard-library imports and general-purpose third-party libraries
import sys
import matplotlib
import numpy as np
import pandas as pd
import torch

# STEP 2: Print library versions for reproducibility
print('System Version:', sys.version)
print('PyTorch version:', torch.__version__)
print('Numpy version:', np.__version__)
print('Pandas version:', pd.__version__)
print('Matplotlib version:', matplotlib.__version__)

The project's own building blocks are imported.

In [ ]:
# STEP 1: Standard-library imports
from pathlib import Path

# STEP 2: Import this project's own building blocks
from utils.dataset_overview import build_slide_table
from utils.mil_data import load_bags
from utils.mil_cv import fold_summary, make_cv_folds
from utils.mil_train import set_seed
from utils.final_classifier import relabel_table
from utils.synthetic_slides import make_synthetic_folds, plot_synthetic_summary, recipe_table, save_synthetic_folds, write_mod_labels

## 2. Constants

Every setting is defined here. `FEATURES_DIR` holds the real tile embeddings and `OUT_DIR` receives the synthetic ones. The folds are set by `N_FOLDS`, `EARLY_STOP_FRACTION` and `SEED`; the notebook that trains on the synthetic slides must make its real folds with the same three values, and it checks that the sources of every synthetic slide belong to the right part of the right fold. The synthetic slides are set by `N_TOTAL` (per fold), `SIZES`, `K_SLIDES` and `TOP_PERCENTAGE`.

In [ ]:
# Major variables
FEATURES_DIR = Path('/mnt/Scratch/phd_streamembed_features/level_1')    # STEP 1: Folder with one subfolder of .pt files per model (real slides, from 01)
OUT_DIR = Path('/mnt/Scratch/phd_streamembed_features/level_1_mod')     # STEP 2: Folder that receives the synthetic vectors (<OUT_DIR>/<MODEL_NAME>/fold_<k>/)
MODEL_NAME = 'UNI2-h'                                                   # STEP 3: Subfolder with the per-slide features
LABELS_FILE = Path('labels/BRACS.xlsx')                                 # STEP 4: Slide labels, patients and official split of the real slides
LABELS_MOD_FILE = Path('labels/BRACS_mod.csv')                          # STEP 5: Label file of the synthetic slides (written here)
SEED = 0                                                                # STEP 6: Seed of every random choice
N_FOLDS, EARLY_STOP_FRACTION = 5, 0.2                                   # STEP 7: Folds; share of the non-held-out slides used to pick the epoch (must equal the values of 02_3)
N_TOTAL, SIZES, K_SLIDES, TOP_PERCENTAGE = 1000, (4096, 8192), 12, 0.20   # STEP 8: Synthetic slides per fold; tiles per synthetic slide; minimum number of source slides; largest share of one source slide
OVERWRITE = False                                                       # STEP 9: Rewrite synthetic files that already exist (default: keep them)
THREE_MAPPING = {'N': 'Non-cancer', 'PB': 'Non-cancer', 'UDH': 'Non-cancer', 'FEA': 'Non-cancer',
                 'ADH': 'Non-invasive atypia/neoplasia', 'DCIS': 'Non-invasive atypia/neoplasia', 'IC': 'Invasive carcinoma'}   # STEP 10: BRACS class -> group
set_seed(SEED)

## 3. Data

### 3.1 Slides and tile embeddings

The bags of tile embeddings (UNI2-h CLS tokens) of the real slides are loaded once; the slide table holds the official split of BRACS.

In [ ]:
# STEP 1: Label table with the official split and the bags of tile embeddings (float16)
table = build_slide_table(LABELS_FILE)
data = load_bags(FEATURES_DIR, MODEL_NAME, table)

### 3.2 The three classes

Non-cancer groups N, PB, UDH and FEA; ADH and DCIS form one class; IC keeps its own class. The slides per class and part are listed.

In [ ]:
# STEP 1: Relabel the table and the bags for the three classes
table3, classes3 = relabel_table(data['table'], THREE_MAPPING)
data3 = {**data, 'table': table3, 'classes': classes3}

# STEP 2: Slides per class and part
pd.crosstab(table3['label'], table3['set']).reindex(classes3)[['Training', 'Validation', 'Testing']]

### 3.3 Cross-validation folds

The official Training and Validation slides are pooled and divided into patient-grouped, class-stratified folds; the Testing slides are in no fold. For every fold a part of the remaining slides is used to select the epoch and the rest to fit the model.

In [ ]:
# STEP 1: The folds (the Testing slides are left out of every fold)
real_folds = make_cv_folds(table3, N_FOLDS, SEED, EARLY_STOP_FRACTION)

# STEP 2: Slides and patients per part, and the held-out slides per class
fold_summary(real_folds, classes3)

## 4. Synthetic slides

### 4.1 Recipes

For every fold, `N_TOTAL` synthetic slides are made: `EARLY_STOP_FRACTION` of them from the real early-stopping slides of the fold and the others from its real fit slides, in equal numbers for the three classes. A synthetic slide of class *c* has 4096 or 8192 tiles, chosen at random. Its number of source slides is drawn uniformly between `K_SLIDES` and the number of real slides of class *c* in that part of the fold; the sources are distinct real slides of class *c*. The tiles taken from each source are drawn with replacement, the number of tiles per source is a random split of the size in which no source gives more than `TOP_PERCENTAGE` of the tiles, and the tiles of all sources are shuffled into one random order. The generator is seeded; the recipes are kept in `OUT_DIR/recipes` and read again on a later run with the same settings and folds, so a repeated run gives the same slides.

In [ ]:
# STEP 1: Recipes of every fold (made now, or read from OUT_DIR/recipes); the tensors are not built yet
folds = make_synthetic_folds(data3, real_folds, N_TOTAL, SIZES, K_SLIDES, TOP_PERCENTAGE, EARLY_STOP_FRACTION, seed=SEED, cache_dir=OUT_DIR / 'recipes')

# STEP 2: Synthetic slides per fold, part and class, and their sources
synthetic = recipe_table(folds)
display(synthetic.groupby(['fold', 'part', 'label']).size().unstack().rename_axis(columns=None))
synthetic[['size', 'n_sources', 'max_share']].describe().round(3)

### 4.2 Properties of the synthetic slides

The rules of the construction are checked when the recipes are made (`check_recipes`); the figure shows how they are used.

In [ ]:
# STEP 1: Figure
fig_synthetic = plot_synthetic_summary(synthetic, TOP_PERCENTAGE, K_SLIDES)

## 5. Saving

Every synthetic slide is assembled from the real bags and written as one file, `OUT_DIR/UNI2-h/fold_<k>/<slide>.pt`, with the keys of the files of notebook 01 (`cls_features`, `coordinates`, `model_encoder`, `tile_params`) and the recipe (`label`, `sources`, `counts`, `source_indices`, `source_ids`). The coordinates are zero because a synthetic tile has no position on a slide, and `tile_params` carries `synthetic=True` and the construction settings. The label file `BRACS_mod.csv` has the columns of `BRACS.xlsx` (the slide name is also its patient identifier, because a synthetic slide is its own group) and, in addition, `fold`, `part`, `size`, `n_sources`, `max_share` and `sources`. Existing files are kept unless `OVERWRITE` is set.

In [ ]:
# STEP 1: Vectors
settings = dict(n_total=N_TOTAL, sizes=list(SIZES), k_slides=K_SLIDES, top_percentage=TOP_PERCENTAGE, stop_fraction=EARLY_STOP_FRACTION, seed=SEED, n_folds=N_FOLDS)
paths = save_synthetic_folds(folds, data3, OUT_DIR, MODEL_NAME, settings, overwrite=OVERWRITE)

# STEP 2: Labels
labels_mod = write_mod_labels(folds, LABELS_MOD_FILE)
print(f'{len(paths)} files under {OUT_DIR / MODEL_NAME}; {len(labels_mod)} rows in {LABELS_MOD_FILE}')
labels_mod.head()

## 6. Verification

One slide per fold is read back and compared with its sources: the tiles must be exactly the tiles of the real bags named by the recipe, every source must belong to the right part of the fold, and no held-out slide may be a source.

In [ ]:
# STEP 1: Reload one slide per fold and compare it with the real bags
for k, fold in enumerate(folds):
    row = labels_mod[labels_mod['fold'] == k].iloc[0]
    saved = torch.load(OUT_DIR / MODEL_NAME / f'fold_{k}' / f"{row['WSI Filename']}.pt", weights_only=True)
    ok_rows = torch.equal(saved['cls_features'][:64], torch.stack([data3['bags'][saved['sources'][int(i)]][int(j)] for i, j in zip(saved['source_ids'][:64], saved['source_indices'][:64])]))
    ok_part = set(saved['sources']) <= set(fold[row['part']])
    ok_held = not set(saved['sources']) & set(fold['held'])
    print(f"fold {k}: {row['WSI Filename']} {tuple(saved['cls_features'].shape)} {saved['cls_features'].dtype} | tiles equal: {ok_rows} | sources in {row['part']}: {ok_part} | no held-out source: {ok_held}")
    assert ok_rows and ok_part and ok_held